# Yeom MEG 3D-reaching — Colab GPU runner

Trains the Brain2Qwerty-style **`convtransformer`** decoder on the Yeom 2023 MEG
reaching dataset using a Colab **NVIDIA GPU**.

**Flow:** mount Drive → download dataset to Drive → load from Drive → train.

**Before running:**
1. *Runtime → Change runtime type → Hardware accelerator: **GPU*** (a T4 is plenty).
2. Make the `hcp_motor_decoder` code available — either upload the folder to your
   Google Drive, or set a git URL (see the *Get the code* cell).

Data + results are written to Drive, so re-running in a fresh session skips the
download.

In [ ]:
import torch
!nvidia-smi -L
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime -> Change runtime type -> GPU'

In [ ]:
# Colab already has torch(+CUDA), numpy, scipy, scikit-learn. Add the rest:
!pip -q install mat73 remotezip mne

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Get the `hcp_motor_decoder` code

Two ways:
- **Git** (`CODE_SOURCE='git'`): clones the repo (its root **is** the package). The
  repo is **private**, so set `GITHUB_TOKEN` to a GitHub PAT with read access.
- **Drive** (`CODE_SOURCE='drive'`): upload your local `hcp_motor_decoder/` folder to
  Drive and point `DRIVE_CODE_DIR` at it.

In [ ]:
import sys, os, subprocess

CODE_SOURCE    = 'git'     # 'git' or 'drive'
GIT_URL        = 'https://github.com/atticus-429/meg-movement-decoder.git'
GITHUB_TOKEN   = ''        # REQUIRED for the private repo (PAT with repo read)
DRIVE_CODE_DIR = '/content/drive/MyDrive/hcp_motor_decoder'

if CODE_SOURCE == 'git':
    assert GITHUB_TOKEN, 'set GITHUB_TOKEN (private repo); or use CODE_SOURCE=drive'
    url = GIT_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
    if not os.path.isdir('/content/code'):
        subprocess.run(['git', 'clone', url, '/content/code'], check=True)
    PKG = '/content/code'          # repo root is the package
else:
    PKG = DRIVE_CODE_DIR
assert os.path.isdir(PKG), f'code folder not found: {PKG}'
if PKG not in sys.path:
    sys.path.insert(0, PKG)
import config, decode, evaluate
from loaders.yeom import load_yeom
print('code OK at', PKG)

## 2. Configure dataset + download target

In [ ]:
# dataset variant: 'ica' (artifact-cleaned, recommended) or 'epoched'
DATA_VARIANT  = 'ica'
FIGSHARE_FILE = {'ica': '41898840', 'epoched': '41898714'}[DATA_VARIANT]
ZIP_URL = f'https://ndownloader.figshare.com/files/{FIGSHARE_FILE}'  # one ~9.3 GB zip

# usable per-subject .mat are extracted here on Drive (persists across sessions)
DRIVE_DATA_DIR = '/content/drive/MyDrive/yeom_meg/data'
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

# which subject(s) to fetch: a SUBSTRING of the .mat filename.
# Leave '' first to just LIST the archive, then set it and re-run the next cell.
SUBJECT_TOKEN = ''

## 3. Download — efficient (only the subject you need)

Uses HTTP range requests to read the archive's file list and pull **only** the
`.mat` matching `SUBJECT_TOKEN` (~0.5–1 GB) instead of the full 9.3 GB. Run once
with `SUBJECT_TOKEN=''` to see the filenames, then set it and re-run.

In [ ]:
import glob
from remotezip import RemoteZip

existing = glob.glob(os.path.join(DRIVE_DATA_DIR, '**', '*.mat'), recursive=True)
if existing:
    print('already on Drive:', [os.path.basename(f) for f in existing])
else:
    with RemoteZip(ZIP_URL) as z:
        mats = [n for n in z.namelist() if n.lower().endswith('.mat')]
        print(f'{len(mats)} .mat files in the archive:')
        for n in mats:
            print('   ', n)
        if not SUBJECT_TOKEN:
            print('\n>> set SUBJECT_TOKEN above to one of these (a substring) and re-run')
        else:
            sel = [n for n in mats if SUBJECT_TOKEN in n]
            assert sel, f"no .mat matched '{SUBJECT_TOKEN}'"
            for n in sel:
                print('downloading', n, '...')
                z.extract(n, DRIVE_DATA_DIR)
            print('saved to', DRIVE_DATA_DIR)

### Fallback — full download (only if the efficient method errors)

Downloads the whole 9.3 GB zip to fast **local** disk (`/content`, not Drive, to
spare your Drive quota), then extracts your subject to Drive. Set `RUN_FALLBACK=True`.

In [ ]:
RUN_FALLBACK = False
if RUN_FALLBACK:
    import zipfile
    assert SUBJECT_TOKEN, 'set SUBJECT_TOKEN first'
    LOCAL_ZIP = f'/content/yeom_{DATA_VARIANT}.zip'
    if not os.path.exists(LOCAL_ZIP):
        os.system(f'wget -c -O "{LOCAL_ZIP}" "{ZIP_URL}"')   # -c resumes if interrupted
    with zipfile.ZipFile(LOCAL_ZIP) as z:
        mats = [n for n in z.namelist() if n.lower().endswith('.mat')]
        sel = [n for n in mats if SUBJECT_TOKEN in n]
        assert sel, f"no .mat matched '{SUBJECT_TOKEN}'"
        for n in sel:
            z.extract(n, DRIVE_DATA_DIR)
            print('extracted', n)

## 4. Load the subject from Drive

In [ ]:
import numpy as np
X, y, sfreq, names = load_yeom(DRIVE_DATA_DIR, subject=SUBJECT_TOKEN or None,
                              sensor_type='all', resample=150, crop=(0.0, 1.5))
print('X', X.shape, '| classes', names, '| counts', np.bincount(y).tolist(), '| sfreq', sfreq)

## 5. Train the conv→transformer (GPU)

With a GPU, full 5-fold CV is feasible (`CV='kfold'`); use `'holdout'` for a quick
single-split pass. The decoder auto-uses CUDA.

In [ ]:
from decode import build_decoder
from evaluate import bootstrap_error_ci, format_confusion
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split

CV = 'kfold'   # 'kfold' (5-fold, GPU) or 'holdout' (single split, fastest)
dec = build_decoder('convtransformer', sfreq=sfreq)

if CV == 'kfold':
    cvs = StratifiedKFold(5, shuffle=True, random_state=0)
    y_pred, y_eval = cross_val_predict(dec, X, y, cv=cvs, n_jobs=1), y
else:
    tr, te = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=0)
    dec.fit(X[tr], y[tr]); y_pred, y_eval = dec.predict(X[te]), y[te]

err, lo, hi = bootstrap_error_ci(y_eval, y_pred)
chance = 100.0 / len(names)
print(f'convtransformer | accuracy {100*(1-err):.1f}%  '
      f'95% CI [{100*(1-hi):.1f}, {100*(1-lo):.1f}]  chance {chance:.1f}%')
cs, cm = format_confusion(y_eval, y_pred, names)
print(cs)

out = '/content/drive/MyDrive/yeom_meg/result_convtransformer.npz'
np.savez(out, y_true=y_eval, y_pred=y_pred, cm=cm, acc=1 - err, classes=names)
print('saved', out)

## 6. (Optional) CSP baseline for comparison

In [ ]:
dec = build_decoder('csp', sfreq=sfreq)
yp = cross_val_predict(dec, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0))
print('csp baseline accuracy:', round(100 * np.mean(yp == y), 1), '%')